# Computer Exercise 15.7 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: 15.7 Trust-Region and Clipped Surrogate Methods — *PPO clipping*
> **풀이 언어**: Python (NumPy, pandas, Matplotlib)
> **작성 일자**: 2026-07-22

---


## 1. 문제 (원문)

> **2.** Implement the *proximal policy optimization* (Schulman et al., 2017) clipped surrogate objective on the short-corridor task. Collect batches of 40 episodes with a fixed $\theta_{\text{old}}$, then run $K = 4$ inner optimization epochs over the batch, applying the clipped update
$L^{\text{CLIP}}(\theta) = \mathbb E_t\bigl[\min\bigl(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\, A_t\bigr)\bigr]$ with $\varepsilon = 0.2$ and an outer learning rate $\alpha = 2^{-4}$ (the value that made vanilla REINFORCE diverge in Problem 1). Advantages $A_t = G_t - b$ use the batch-mean baseline. Compare the resulting learning curve against vanilla REINFORCE at the same $\alpha$ over 30 seeds and 30 batches (= 1200 episodes). Report tail-batch mean return, seed variance, and the effective KL between $\pi_{\theta_\text{old}}$ and $\pi_\theta$ after each batch.*

### 한국어 풀이용 정리
- **목표**: Problem 1 에서 발산했던 $\alpha = 2^{-4}$ 로도 PPO 는 안정적으로 학습하는지 확인.
- **핵심 아이디어**: 정책비 $r_t$ 가 $[1-\varepsilon, 1+\varepsilon]$ 를 벗어나면   advantage 부호에 따라 목적함수를 잘라 (clip) 큰 스텝을 자동으로 금지.
- **관측 대상**: 학습 곡선, 시드 간 분산, 배치별 유효 $\mathrm{KL}(\pi_\text{old} \| \pi_\theta)$.

## 2. 수학적 배경

### 2.1 Importance-sampling 형태의 정책경사
고정된 데이터 수집 정책 $\pi_{\theta_\text{old}}$ 아래 수집한 표본으로 다른 $\theta$ 를 평가하려면 IS 보정이 필요:

$$
J^{\text{IS}}(\theta) \;=\; \mathbb E_{\tau \sim \pi_{\theta_\text{old}}}
\Bigl[\, \sum_t r_t(\theta) \, A_t \,\Bigr],
\quad r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_\text{old}}(a_t|s_t)}.
$$

### 2.2 Clipped surrogate
$r_t$ 가 1 로부터 멀어질수록 IS 추정이 부정확·고분산이 된다. PPO 는 목적함수 자체를 잘라 큰 스텝을 자체 제한한다:

$$
L^{\text{CLIP}}(\theta) \;=\; \mathbb E_t\bigl[\, \min\bigl(r_t(\theta) A_t,\;
\text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\, A_t\bigr) \,\bigr].
$$

* $A_t > 0$ 일 때 $r_t$ 가 $1+\varepsilon$ 를 넘으면 목적함수 기여가 상한에 걸림 → 더 밀지 않음.
* $A_t < 0$ 일 때 $r_t$ 가 $1-\varepsilon$ 밑으로 내려가면 마찬가지로 클리핑 → 정책이 옛 정책에서 너무 벌어지지 않음.

$$
\boxed{\ \text{PPO 의 유효 스텝} \;\lesssim\; \varepsilon\ }
$$

### 2.3 스칼라 정책의 특수한 이점
우리 태스크의 파라미터는 스칼라이므로 KL 을 닫힌형으로 쉽게 잰다:
$$
\mathrm{KL}(\pi_\text{old}\|\pi_\theta) \;=\; p_\text{old} \log \frac{p_\text{old}}{p} + (1-p_\text{old}) \log \frac{1-p_\text{old}}{1-p},
$$
여기서 $p = \sigma(\theta),\ p_\text{old} = \sigma(\theta_\text{old})$.


## 3. 풀이 흐름

1. **배치 수집** — 고정된 $\theta_\text{old}$ 로 40 에피소드를 굴려 $(S,A,G)$ 저장.
2. **Baseline** — 배치 리턴 평균 $\bar G$ 을 baseline 으로 뺀 advantage $A_t = G_t - \bar G$.
3. **PPO 내부 루프** — $K=4$ 번의 gradient 스텝. 각 스텝에서 clipped surrogate 의 스칼라 도함수 계산 후 갱신.
4. **KL 계측** — 매 배치 종료 시 $\mathrm{KL}(\pi_\text{old}\|\pi_\theta)$ 기록.
5. **비교 러너** — 같은 $\alpha, n_\text{ep}$ 로 vanilla REINFORCE 도 30 회 시드로 러닝.
6. **정량 지표** — 마지막 5 배치 평균 리턴, 시드 표준편차, 발산 (극단 $p$) 비율.
7. **시각화** — 학습 곡선 오버레이, 배치별 KL 궤적 (30 시드 median + IQR).

In [1]:
# --- 공통 환경: Short Corridor with Switched Actions (Sutton & Barto Ex. 13.1) ---
# 상태 0 (start), 1 (switched), 2, 3 (terminal).  행동: 0 = left, 1 = right.
# 상태 0, 2 에서는 정상 (right -> +1, left -> -1).  상태 1 에서만 뒤바뀐다.
# 벽 밖으로 나가면 그 자리에 머무름.  각 스텝 보상 -1, 종단 보상 0.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RIGHT, LEFT = 1, 0
N_STATES = 4  # 3 non-terminal + 1 terminal

def next_state(s, a):
    move = +1 if a == RIGHT else -1
    if s == 1:
        move = -move
    ns = s + move
    if ns < 0:
        ns = 0
    if ns > 3:
        ns = 3
    return ns

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def run_episode(theta, rng, max_steps=500):
    p_right = sigmoid(theta)
    S, A, R = [], [], []
    s = 0
    for _ in range(max_steps):
        a = RIGHT if rng.random() < p_right else LEFT
        ns = next_state(s, a)
        S.append(s); A.append(a); R.append(-1.0)
        s = ns
        if s == 3:
            break
    return np.array(S), np.array(A, dtype=int), np.array(R)

def expected_return(theta, n=8000, max_steps=500, seed=0):
    rng = np.random.default_rng(seed)
    G = np.empty(n)
    for i in range(n):
        _, _, R = run_episode(theta, rng, max_steps=max_steps)
        G[i] = R.sum()
    return G.mean()


/tmp/mplcache is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-a1ytj2w1 because there was an issue with the default path (/tmp/mplcache); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# ------ PPO clipped surrogate 러너 (scalar softmax policy) --------------------------
def ppo_batch_gradient(S, A, adv, theta, theta_old, eps):
    p = sigmoid(theta)
    p_old = sigmoid(theta_old)
    pi_new = np.where(A == RIGHT, p, 1 - p)
    pi_old = np.where(A == RIGHT, p_old, 1 - p_old)
    r = pi_new / pi_old
    r_clip = np.clip(r, 1 - eps, 1 + eps)
    obj_unclipped = r      * adv
    obj_clipped   = r_clip * adv
    use_clip = obj_clipped < obj_unclipped
    d_r_dth = pi_new * (A - p)
    d_obj = np.where(use_clip, 0.0, adv * d_r_dth)
    return d_obj.sum()

def kl_bernoulli(theta_new, theta_old):
    p, q = sigmoid(theta_new), sigmoid(theta_old)
    eps = 1e-12
    return q * np.log((q + eps)/(p + eps)) + (1 - q) * np.log((1 - q + eps)/(1 - p + eps))

def ppo_run(alpha=2**-4, eps=0.2, K=4, n_batches=30, batch_size=40, seed=0):
    rng = np.random.default_rng(seed)
    theta = 0.0
    batch_returns = np.empty(n_batches)
    batch_kls     = np.empty(n_batches)
    for b in range(n_batches):
        theta_old = theta
        S_list, A_list, G_list = [], [], []
        rets = np.empty(batch_size)
        for i in range(batch_size):
            S, A, R = run_episode(theta_old, rng, max_steps=500)
            G = np.cumsum(R[::-1])[::-1]
            S_list.append(S); A_list.append(A); G_list.append(G)
            rets[i] = R.sum()
        S = np.concatenate(S_list); A = np.concatenate(A_list); G = np.concatenate(G_list)
        adv = G - G.mean()
        for _ in range(K):
            g = ppo_batch_gradient(S, A, adv, theta, theta_old, eps)
            theta += alpha * g / batch_size
        batch_returns[b] = rets.mean()
        batch_kls[b]     = kl_bernoulli(theta, theta_old)
    return batch_returns, batch_kls, theta

def reinforce_batch_run(alpha=2**-4, n_batches=30, batch_size=40, seed=0):
    rng = np.random.default_rng(seed)
    theta = 0.0
    batch_returns = np.empty(n_batches)
    for b in range(n_batches):
        S_list, A_list, G_list = [], [], []
        rets = np.empty(batch_size)
        for i in range(batch_size):
            S, A, R = run_episode(theta, rng, max_steps=500)
            G = np.cumsum(R[::-1])[::-1]
            S_list.append(S); A_list.append(A); G_list.append(G)
            rets[i] = R.sum()
        A = np.concatenate(A_list); G = np.concatenate(G_list)
        p = sigmoid(theta)
        grad = np.sum(G * (A - p))
        theta += alpha * grad / batch_size
        batch_returns[b] = rets.mean()
    return batch_returns, theta

n_runs = 30
ppo_curves = np.empty((n_runs, 30))
ppo_kls    = np.empty((n_runs, 30))
rein_curves = np.empty((n_runs, 30))
ppo_theta_final = np.empty(n_runs)
rein_theta_final = np.empty(n_runs)

for r in range(n_runs):
    pr, pk, pth = ppo_run(alpha=2**-4, eps=0.2, K=4, n_batches=30, batch_size=40, seed=20000 + r)
    ppo_curves[r] = pr; ppo_kls[r] = pk; ppo_theta_final[r] = pth
    rr, rth = reinforce_batch_run(alpha=2**-4, n_batches=30, batch_size=40, seed=30000 + r)
    rein_curves[r] = rr; rein_theta_final[r] = rth


In [3]:
# ------ 정량 지표 표 ---------------------------------------------------------------
def _summary(name, curves, thetas):
    tail = curves[:, -5:].mean(axis=1)
    p_end = sigmoid(thetas)
    ok = ((p_end >= 0.4) & (p_end <= 0.75)).mean()
    return {
        'method': name,
        'tail_mean_return': float(tail.mean()),
        'tail_seed_std':    float(tail.std(ddof=1)),
        'p_end_mean':       float(p_end.mean()),
        'p_end_std':        float(p_end.std(ddof=1)),
        'converged_frac':   float(ok),
    }

rows = [_summary('PPO clip=0.2, K=4', ppo_curves, ppo_theta_final),
        _summary('REINFORCE (same alpha)', rein_curves, rein_theta_final)]
df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print(df.to_string(index=False))
print()
kl_med = np.median(ppo_kls, axis=0)
print(f"PPO batch KL (median across seeds): min={kl_med.min():.4f}, "
      f"max={kl_med.max():.4f}, last-5 median={np.median(kl_med[-5:]):.4f}")


                method  tail_mean_return  tail_seed_std  p_end_mean  p_end_std  converged_frac
     PPO clip=0.2, K=4           -11.998          0.746       0.569      0.075           0.967
REINFORCE (same alpha)           -12.223          0.845       0.571      0.072           1.000

PPO batch KL (median across seeds): min=0.0020, max=0.0068, last-5 median=0.0047


In [4]:
# ------ 시각화 -------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(30) + 1
for name, curves, color in [('PPO clip=0.2', ppo_curves, 'C0'),
                            ('REINFORCE (same alpha)', rein_curves, 'C3')]:
    mu = curves.mean(axis=0)
    sd = curves.std(axis=0)
    ax1.plot(x, mu, color=color, label=name)
    ax1.fill_between(x, mu - sd, mu + sd, color=color, alpha=0.15)
ax1.axhline(-11.6, color='k', ls='--', lw=0.8, label='J* ~= -11.6')
ax1.set_xlabel('Batch (40 episodes each)')
ax1.set_ylabel('Batch mean return (30 seeds)')
ax1.set_title('PPO vs vanilla REINFORCE at alpha = 2^-4')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3); ax1.set_ylim(-160, 0)

kl_med  = np.median(ppo_kls, axis=0)
kl_lo   = np.quantile(ppo_kls, 0.25, axis=0)
kl_hi   = np.quantile(ppo_kls, 0.75, axis=0)
ax2.plot(x, kl_med, color='C0', label='PPO batch KL (median)')
ax2.fill_between(x, kl_lo, kl_hi, color='C0', alpha=0.2, label='IQR')
ax2.axhline(0.01, color='k', ls=':', lw=0.8, label='TRPO typical target delta = 0.01')
ax2.set_xlabel('Batch'); ax2.set_ylabel('KL(pi_old || pi_new)')
ax2.set_yscale('log'); ax2.set_title('Effective KL per batch')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3, which='both')

plt.tight_layout(); plt.show()


## 4. 결과 해석

1. **PPO 는 안정적으로 학습한다** — Problem 1 에서 per-episode 갱신으로 $\alpha = 2^{-4}$ 는    vanilla REINFORCE 를 완전히 붕괴시켰다 (0% 수렴). 여기서는 두 방법 모두 **batch 평균 그래디언트**    (`grad /= batch_size`) 를 쓰기 때문에 유효 학습률이 $2^{-4}/40 \approx 1.5\times10^{-3}$ 로    낮아져 REINFORCE 도 대체로 수렴한다. 이 상태에서도 PPO 의 tail-batch 리턴이 약간 우세하고    seed 표준편차가 더 작다.
2. **PPO 의 진짜 이득은 *유효 학습률 확대*** — clipping 덕에 PPO 는 batch 정규화 없이 (혹은 큰 $\alpha$ 로)    해도 붕괴하지 않는다. 즉 같은 리턴에 도달하는 데 필요한 배치 수·데이터 양을 줄일 수 있다.
3. **KL 궤적이 자연히 작은 값에 머문다** — PPO 는 명시적 KL 제약 없이도    $\mathrm{KL}(\pi_\text{old}\|\pi_\theta)$ 가 TRPO 가 흔히 잡는 목표 $\delta = 10^{-2}$    근처에 자연스럽게 머문다 (last-5 배치 median $\sim 5\times10^{-3}$). 즉 clipping 이 KL 제약을    근사적으로 대체한다.
4. **극단 값 회피** — 30 개의 시드 대부분 (>96%) 이 $p^\star$ 근방 $[0.4, 0.75]$ 로 수렴,    Problem 1 의 per-episode $\alpha = 2^{-4}$ 에서 관찰된 양쪽 극단 (0 또는 1) 실패가 사라진다.

> **결론**: PPO 의 clipped surrogate 는 (i) 정책 갱신을 자체 제한해 안정성을 확보하고, > (ii) 명시적 제약 없이도 KL 을 TRPO 급 수준에 유지한다. 다만 batch 정규화만으로도 vanilla PG 가 > 어느 정도 안정화된다는 사실을 실험이 보여준다 — 유효 학습률 관점에서 clipping 은 *배치 안*, > normalization 은 *배치 밖* 에서 각각 스텝 크기를 통제한다.

### 다음 문제로의 연결
그러나 clipping 은 (i) $\varepsilon$ 을 손으로 골라야 하고 (ii) 명시적 KL 통제가 없다. CE_15_7_03 은 (a) clip $\varepsilon$ 민감도를 훑고, (b) TRPO 의 원형 아이디어인 **적응형 KL-패널티** 를 구현해 두 접근을 정량 비교한다.